# Analisis Integral de la Dinamica Monetaria y Cambiaria de Argentina
### Un Estudio Intertemporal del Impacto de las Politicas de Estabilizacion y la Deuda (2016-2017 vs. 2024-2025)

#### 1. Introduccion y Justificacion del Estudio
La macroeconomia argentina contemporanea se caracteriza por la recurrencia de desequilibrios estructurales y cambios drasticos en los regimenes de politica monetaria y cambiaria. A menudo, las estrategias diseñadas para contener la nominalidad y estabilizar el sistema financiero conllevan severas implicancias en la distribucion del ingreso y en el endeudamiento de la nacion.

Este estudio realiza un contraste analitico y empirico entre dos esquemas monetarios opuestos aplicados en la ultima decada:
- **La gestion 2016-2017:** Caracterizada por la desregulacion cambiaria inicial, el regimen de metas de inflacion mediante altas tasas reales de interes, y una fuerte expansion del endeudamiento externo internacional que desemboco en la crisis de 2018 y el subsecuente programa de asistencia Stand-By con el Fondo Monetario Internacional (FMI).
- **La gestion 2024-2025:** Caracterizada por un programa de estabilizacion de shock sustentado en superavit fiscal financiero, crawling peg al 2% mensual, emision cero de la base monetaria, y la licuacion de los pasivos del Banco Central y de los ingresos de la poblacion mediante tasas de interes reales negativas.

A traves de la unificacion y procesamiento de datos provistos por el Banco Central (BCRA), la Comision Nacional de Valores (CNV) y la Bolsa y Mercados Argentinos (BYMA), analizaremos de que manera las politicas monetarias de ambos periodos impactaron sobre el ahorro en moneda local (depósitos), el mercado corporativo de capitales (Obligaciones Negociables) y, centralmente, sobre el poder adquisitivo real de los saldos monetarios de la poblacion.

## 2. Contexto Macroeconomico e Institucional

Para comprender la dinamica de las series temporales, es indispensable caracterizar los marcos conceptuales de ambas politicas economicas y la continuidad de sus principales actores tecnicos:

### 2.1 El Regimen de Metas de Inflacion (2016-2017)
Bajo la direccion del Banco Central conducido por Federico Sturzenegger y la posterior gestion de Luis Caputo en el Ministerio de Finanzas, la administracion implemento un esquema de metas de inflacion con flotacion cambiaria. La politica monetaria operaba elevando la tasa nominal de interes (LEBAC) en terminos reales para desalentar la demanda agregada y esterilizar la emision. Este enfoque de la **Escuela Monetarista** asumia que la inflacion es un fenomeno estrictamente monetario controlable mediante el precio del dinero. Sin embargo, la persistencia del deficit fiscal y la acumulacion de pasivos de corto plazo en el Banco Central derivaron en una perdida de credibilidad, una masiva reversion de los flujos de capitales (sudden stop) en 2018 y la contratacion de un prestamo Stand-By historico con el FMI por USD 57.000 millones para evitar el default.

### 2.2 El Programa de Estabilizacion de Shock y Licuacion (2024-2025)
Nuevamente con la presencia de Luis Caputo (como Ministro de Economia) y Santiago Bausili (en la presidencia del Banco Central), la administracion de Javier Milei inicio un plan de estabilizacion enfocado en el ancla cambiaria (crawling peg al 2% mensual) y el saneamiento del balance del Banco Central mediante la emision de base monetaria cero. Metodologicamente, se recurrio a una profunda **licuacion de saldos reales**: las tasas de interes pasivas (BADLAR) se mantuvieron sistematicamente por debajo del ritmo de inflacion mensual de shock (tasas reales negativas). Este mecanismo licuo de forma acelerada los pasivos remunerados del Banco Central y licuo asimismo los ingresos y sueldos reales de la poblacion, traduciendose en una marcada desmonetizacion del consumo interno. Simultaneamente, la gestion se concentro en refinanciar y obtener rollover de la deuda contraida con el FMI en 2018.

In [ ]:
# Importacion de librerias de procesamiento, visualizacion y analisis numerico
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Configuracion estetica de graficos para reportes macrofinancieros senior
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

## 3. Carga y Consolidacion de Datos (Data Wrangling)

Procedemos a preparar el entorno y consolidar los archivos de las series temporales.

### 3.1 Deteccion de Entorno y Carga de Datasets (Soporte Google Colab)
Para asegurar la total portabilidad y ejecucion del cuaderno en entornos locales o en la nube de Google Colab, implementamos un cargador dinamico interactivo que intenta clonar automaticamente el repositorio de GitHub y ofrece una carga interactiva como fallback.

In [ ]:
# Deteccion automatica de Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

datasets_dir = "datasets"

if IN_COLAB:
    print("--- Entorno Google Colab detectado ---")
    import os
    
    # Intentar clonar automaticamente los datasets desde el repositorio de GitHub del proyecto
    if not os.path.exists(datasets_dir):
        repo_name = "argentina-monetary-macro-analysis"
        try:
            print("Intentando clonar datasets de forma automatica desde el repositorio de GitHub...")
            os.system(f"git clone https://github.com/joaquinescalante/{repo_name}.git temp_repo")
            if os.path.exists("temp_repo/datasets"):
                os.rename("temp_repo/datasets", datasets_dir)
                print("Datasets clonados y organizados exitosamente en la carpeta 'datasets/'.")
            os.system("rm -rf temp_repo")
        except Exception as e:
            print("No se pudo clonar automaticamente el repositorio.")
            
    # Fallback: Carga interactiva si la clonacion automatica falla o no tiene los archivos
    if not os.path.exists(datasets_dir) or len(os.listdir(datasets_dir)) == 0:
        if not os.path.exists(datasets_dir):
            os.makedirs(datasets_dir)
        from google.colab import files
        print("Por favor, suba de forma manual los archivos CSV correspondientes a los datasets del proyecto:")
        uploaded = files.upload()
        
        # Reubicar los archivos cargados en el directorio local datasets/
        for filename in uploaded.keys():
            destino = os.path.join(datasets_dir, filename)
            if os.path.exists(destino):
                os.remove(destino)
            os.rename(filename, destino)
        print("\nDatasets cargados e interactivamente organizados en la carpeta 'datasets/'.")
else:
    print("--- Entorno local detectado. Los datasets se cargaran directamente desde la carpeta 'datasets/' ---")

In [ ]:
# Carga de archivos temporales mensuales consolidados
df_ipc = pd.read_csv(os.path.join(datasets_dir, "Indice IPC-IPCGA empalmado.csv"))
df_tasas = pd.read_csv(os.path.join(datasets_dir, "TASA BADLAR Y ADELANTOS.csv"))
df_dep_usd = pd.read_csv(os.path.join(datasets_dir, "volumen de deposito- USD.csv"))
df_dep_ars = pd.read_csv(os.path.join(datasets_dir, "volumen de depositos - ars.csv"))
df_byma = pd.read_csv(os.path.join(datasets_dir, "Volumen operado de mercado BYMa-IAMC.xlsx - Hoja2.csv"))
df_cpi_us = pd.read_csv(os.path.join(datasets_dir, "Consumer Price Index for All Urban Consumers (CPU-)EEUU - Hoja 1.csv"))

# Carga del registro de emisiones de ONs (Frecuencia diaria)
df_on = pd.read_csv(os.path.join(datasets_dir, "ON para base de datos.csv"))

print("Datasets cargados exitosamente en memoria.")

In [ ]:
# Estandarizacion de formato de fecha en todos los DataFrames
dfs_mensuales = [df_ipc, df_tasas, df_dep_usd, df_dep_ars, df_byma, df_cpi_us]
for df in dfs_mensuales:
    df['fecha'] = pd.to_datetime(df['fecha'])

df_on['Fecha'] = pd.to_datetime(df_on['Fecha'])

# Consolidacion de las series mensuales mediante merge
df_macro = df_ipc.copy()
for df in dfs_mensuales[1:]:
    df_macro = pd.merge(df_macro, df, on='fecha', how='outer')

df_macro = df_macro.sort_values('fecha').reset_index(drop=True)
print(f"Dimensiones del DataFrame macroeconomico consolidado: {df_macro.shape}")

### 3.2 Inspeccion Inicial y Estadisticas del Dataset Consolidado
Exhibimos las primeras observaciones consolidadas del dataset macroeconomico y evaluamos sus estadisticas descriptivas principales.

In [ ]:
# Visualizar las primeras 5 filas de las columnas mas relevantes
columnas_muestrario = ['fecha', 'IPC', 'Tasa_BADLAR_promedio_mensual_TNA', 'Saldo_Plazo_Fijo_PESOS_residente_miles_ARS', 'CPI']
df_macro[columnas_muestrario].head()

In [ ]:
# Resumen estadistico descriptivo de las variables clave
df_macro[columnas_muestrario[1:]].describe().T

## 4. Limpieza de Datos y Tratamiento de Outliers

Procedemos a identificar la existencia de duplicados y nulos. Asimismo, analizamos los valores extremos (outliers) macroeconomicos.

In [ ]:
# 4.1 Verificacion y Tratamiento de Duplicados y Nulos
duplicados = df_macro.duplicated(subset=['fecha']).sum()
nulos_totales = df_macro.isnull().sum().sum()
print(f"Registros duplicados por fecha: {duplicados}")
print(f"Valores faltantes en el dataset consolidado: {nulos_totales}")

if nulos_totales > 0:
    # Interpolacion lineal con ffill y bfill para evitar nulos en extremos de la serie
    df_macro = df_macro.interpolate(method='linear').ffill().bfill()
    print("Imputacion lineal y propagacion de extremos realizada.")

In [ ]:
# 4.2 Analisis de Outliers en Variables Clave
variables_interes = ['Tasa_BADLAR_promedio_mensual_TNA', 'Saldo_Plazo_Fijo_PESOS_residente_miles_ARS']

for var in variables_interes:
    Q1 = df_macro[var].quantile(0.25)
    Q3 = df_macro[var].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = df_macro[(df_macro[var] < limite_inferior) | (df_macro[var] > limite_superior)]
    print(f"Outliers detectados en '{var}' (Linf: {limite_inferior:.2f}, Lsup: {limite_superior:.2f}): {len(outliers)}")
    if len(outliers) > 0:
        print(outliers[['fecha', var]])

# Nota Metodologica: Los outliers identificados no obedecen a fallas de medicion sino a shocks macros y
# devaluaciones bruscas reales. En el contexto de la Economia Conductual, la presencia de estos saltos
# condiciona las expectativas no lineales de los agentes financieros. Mantener estos registros es crucial
# para entrenar estimadores en regimenes de alta volatilidad.

## 5. Transformaciones de Datos y Feature Engineering

Para unificar y comparar los periodos de analisis, desarrollaremos las siguientes variables macroeconomicas derivadas:
1. **Tipo de Cambio Implicito Mensual (TCI):** Estimado a partir del ratio entre la emision corporativa en Pesos y Dolares de Obligaciones Negociables. Desde la perspectiva de la **Escuela Austriaca**, este precio representa el canal libre de asignacion de capitales que eluden las restricciones cambiarias (cepo).
2. **Tasa de Inflacion Mensual de Argentina y EEUU:** Variacion porcentual mensual de los indices de precios correspondientes.
3. **Tasas de Interes Reales Efectivas Mensuales (TEM Real):** Deflactadas por la inflacion del periodo mediante la ecuacion de Fisher.
4. **Agregacion y Deflactacion de Depositos:** Saldo total de depositos en pesos y dolares expresados en terminos reales de poder adquisitivo constante (base Diciembre 2016 = 100).

In [ ]:
# 5.1 Estimacion del Tipo de Cambio Implicito Corporativo (ON)
df_on_usd = df_on[df_on['Moneda'].str.upper() == 'USD'].copy()
df_on_usd['TC_implicito'] = (df_on_usd['MONTO_ARS_EMISION_MILLONES'] * 1_000_000) / df_on_usd['Monto_emision']

# Agrupar por mes y calcular el promedio del tipo de cambio implicito corporativo
df_on_usd['fecha_mensual'] = df_on_usd['Fecha'].dt.to_period('M').dt.to_timestamp()
df_tc_mensual = df_on_usd.groupby('fecha_mensual')['TC_implicito'].mean().reset_index()
df_tc_mensual.rename(columns={'fecha_mensual': 'fecha', 'TC_implicito': 'tipo_cambio_implicito'}, inplace=True)

# Acoplar al DataFrame principal
df_macro = pd.merge(df_macro, df_tc_mensual, on='fecha', how='left')

# Imputacion robusta de meses sin emisiones mediante forward y backward fill
df_macro['tipo_cambio_implicito'] = df_macro['tipo_cambio_implicito'].ffill().bfill()
print("Tipo de cambio implicito de ONs acoplado e imputado.")

In [ ]:
# 5.2 Calculo de Tasas de Inflacion Mensual
# Variacion porcentual mensual del IPC argentino
df_macro['inflacion_mensual_arg'] = df_macro['Índice_Empalmado _(Dic-16=100)'].pct_change()
# Variacion porcentual mensual del CPI de EEUU
df_macro['inflacion_mensual_usa'] = df_macro['CPI'].pct_change()

# Completar el primer registro con la media historica para evitar nulos
df_macro['inflacion_mensual_arg'] = df_macro['inflacion_mensual_arg'].fillna(0.02)
df_macro['inflacion_mensual_usa'] = df_macro['inflacion_mensual_usa'].fillna(0.001)

### 5.3 Calculo de Tasas de Interes Reales (Ecuacion de Fisher)

Para deflactar las tasas de interes nominales (TNA), primero convertimos la TNA a una tasa efectiva mensual nominal ($TEM_{\text{nominal}} = TNA / 12$) y aplicamos la formula de Fisher para obtener la tasa efectiva mensual real ($TEM_{\text{real}}$):

$$1 + TEM_{\text{real}} = \frac{1 + TEM_{\text{nominal}}}{1 + \text{Inflación}_{\text{mensual}}}$$

Posteriormente, anualizamos la tasa real para que sea comparable con la TNA nominal de referencia:

$$TNA_{\text{real}} = TEM_{\text{real}} \times 12$$

**Canal de Transmision y Licuacion:** Las tasas nominales fijadas sistematicamente por debajo de la inflacion representan una transferencia real de riqueza desde los acreedores (ahorristas y asalariados tenedores de pesos) hacia el deudor principal (Banco Central y Tesoro Nacional), licuando el valor real de la riqueza liquida de los hogares.

In [ ]:
# Convertimos la TNA a tasa efectiva mensual nominal (TEM)
df_macro['TEM_BADLAR'] = df_macro['Tasa_BADLAR_promedio_mensual_TNA'] / 12
df_macro['TEM_Adelantos'] = df_macro['Tasa_Adelantos_Promedio_mensual_TNA'] / 12

# Ecuacion de Fisher
df_macro['TEM_BADLAR_Real'] = ((1 + df_macro['TEM_BADLAR']) / (1 + df_macro['inflacion_mensual_arg'])) - 1
df_macro['TEM_Adelantos_Real'] = ((1 + df_macro['TEM_Adelantos']) / (1 + df_macro['inflacion_mensual_arg'])) - 1

# Tasa anualizada real equivalente (TNA Real)
df_macro['TNA_BADLAR_Real'] = df_macro['TEM_BADLAR_Real'] * 12
df_macro['TNA_Adelantos_Real'] = df_macro['TEM_Adelantos_Real'] * 12
print("Tasas reales calculadas exitosamente.")

In [ ]:
# 5.4 Deflactacion de Depositos a Pesos Constantes
# Agrupacion de saldos nominales de depositos de residentes en ARS
df_macro['depositos_totales_ARS_nom'] = (
    df_macro['Saldo_Cta_Cte_PESOS_residentes_miles_ARS'] +
    df_macro['Saldo_Caja_Ahorro_PESOS_residentes_miles_ARS'] +
    df_macro['Saldo_Plazo_Fijo_PESOS_residente_miles_ARS']
)

# Deflactamos por el Indice Empalmado (Dic-16 = 100) para obtener pesos constantes de poder adquisitivo
df_macro['depositos_totales_ARS_real'] = (df_macro['depositos_totales_ARS_nom'] / df_macro['Índice_Empalmado _(Dic-16=100)']) * 100

# Agrupacion de saldos nominales de depositos de residentes en USD (Valuados en pesos)
df_macro['depositos_totales_USD_val_ARS'] = (
    df_macro['Saldo_Cta_Cte_USD_residente_miles_ARS'] +
    df_macro['Saldo_Caja_Ahorro_USD_residente_miles_ARS'] +
    df_macro['Saldo_Plazo_Fijo_USD_residente_miles_ARS']
)

# Convertimos a USD fisicos dividiendo por el tipo de cambio implicito
df_macro['depositos_totales_USD_real'] = df_macro['depositos_totales_USD_val_ARS'] / df_macro['tipo_cambio_implicito']
print("Saldos agregados de depositos deflactados calculados.")

In [ ]:
# 5.5 Identificacion del Periodo de Gestion
df_macro['gestion'] = np.where(df_macro['fecha'].dt.year <= 2017, 'Macri (2016-2017)', 'Milei (2024-2025)')
print(df_macro['gestion'].value_counts())

## 6. Analisis Exploratorio de Datos (EDA) Comparativo

Procedemos a graficar y contrastar las principales dinamicas monetarias y financieras de ambos periodos economicos, incorporando rotulos y comentarios explicativos de alta finanza.

In [ ]:
# 6.1 Evolucion del Tipo de Cambio Implicito y su Tasa de Devaluacion Mensual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (gest, data) in enumerate(df_macro.groupby('gestion')):
    ax = axes[i]
    # Normalizamos el tipo de cambio al inicio de cada periodo para observar la devaluacion acumulada relativa
    tc_inicial = data['tipo_cambio_implicito'].iloc[0]
    tc_normalizado = (data['tipo_cambio_implicito'] / tc_inicial - 1) * 100
    
    sns.lineplot(x=data['fecha'].dt.strftime('%m-%Y'), y=tc_normalizado, ax=ax, marker='o', color='navy')
    ax.set_title(f"Devaluacion Acumulada del Tipo de Cambio Implicito: {gest}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Variacion Acumulada (%)")
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 6.2 Tasas de Interes Nominales vs Reales (BADLAR)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (gest, data) in enumerate(df_macro.groupby('gestion')):
    ax = axes[i]
    # TNA nominal vs TNA real anualizada
    sns.lineplot(x=data['fecha'].dt.strftime('%m-%Y'), y=data['Tasa_BADLAR_promedio_mensual_TNA']*100, 
                 ax=ax, marker='o', label='BADLAR Nominal (TNA)', color='royalblue')
    sns.lineplot(x=data['fecha'].dt.strftime('%m-%Y'), y=data['TNA_BADLAR_Real']*100, 
                 ax=ax, marker='s', label='BADLAR Real (Anual)', color='crimson')
    
    ax.axhline(0, color='black', linestyle='--', alpha=0.7)
    ax.set_title(f"Tasas de Interes Pasivas (BADLAR): {gest}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Tasa (%)")
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Evolucion Real de los Depositos del Sistema Financiero
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (gest, data) in enumerate(df_macro.groupby('gestion')):
    ax = axes[i]
    ax2 = ax.twinx()
    
    sns.lineplot(x=data['fecha'].dt.strftime('%m-%Y'), y=data['depositos_totales_ARS_real']/1e6, 
                 ax=ax, marker='o', color='teal', label='Depositos ARS Reales (millones Dic-16)')
    sns.lineplot(x=data['fecha'].dt.strftime('%m-%Y'), y=data['depositos_totales_USD_real']/1e3, 
                 ax=2, marker='s', color='orange', label='Depositos USD Reales (millones)')
    
    ax.set_title(f"Monetizacion Bancaria Real: {gest}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Pesos Constantes (Millones)", color='teal')
    ax2.set_ylabel("Dolares Fisicos (Millones)", color='orange')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Matriz de Correlacion Cruzada de Spearman
variables_corr = [
    'tipo_cambio_implicito', 'inflacion_mensual_arg', 
    'Tasa_BADLAR_promedio_mensual_TNA', 'TNA_BADLAR_Real',
    'depositos_totales_ARS_real', 'depositos_totales_USD_real',
    'Volumen_Total_ON_en_millones_ARS'
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for i, (gest, data) in enumerate(df_macro.groupby('gestion')):
    corr_matrix = data[variables_corr].corr(method='spearman')
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[i], vmin=-1, vmax=1)
    axes[i].set_title(f"Matriz de Correlacion de Spearman: {gest}")

plt.tight_layout()
plt.show()

## 7. Seleccion de Variables y Particion para Modelos

Para estructurar modelos predictivos de Machine Learning sin incurrir en sesgos nominales, procedemos al escalado de variables y la division estratificada del conjunto de datos.

In [ ]:
# Seleccion de variables predictoras e hidrograficas de estabilidad
features = [
    'inflacion_mensual_arg', 'inflacion_mensual_usa', 
    'Tasa_BADLAR_promedio_mensual_TNA', 'Tasa_Adelantos_Promedio_mensual_TNA',
    'Volumen_Total_ON_en_millones_ARS'
]

# Target: BADLAR Real
target = 'TNA_BADLAR_Real'

X = df_macro[features]
y = df_macro[target]

# Escalado con MinMaxScaler para amortiguar el efecto de las diferencias en la nominalidad
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# Particion de entrenamiento (80%) y evaluacion (20%) con estratificacion por gestion
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.20, random_state=42, stratify=df_macro['gestion']
)

print(f"Dimensiones de Entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de Evaluacion (X_test): {X_test.shape}")

## 8. Conclusiones y Observaciones Finales

El analisis comparativo e intertemporal de las dinamicas macrofinancieras de Argentina expone limites estructurales en ambos enfoques de politica economica:

1. **El Costo de la Estabilizacion y la Distribucion Real:**
   - En el periodo **2016-2017**, las tasas reales fuertemente positivas actuaron como incentivo para sostener los saldos en pesos, pero requirieron un volumen insostenible de emision de pasivos remunerados (LEBAC) y la de la acumulacion de deuda externa. La posterior crisis forzo la asistencia del FMI en 2018.
   - En el periodo **2024-2025**, el programa de estabilizacion logro frenar la nominalidad extrema a costa de una sensible caida de los saldos de depósitos reales en la etapa inicial de shock y una profunda licuacion del poder adquisitivo de los salarios reales y del ahorro minorista (provocada por la tasa real negativa de interes).
2. **Continuidad de Actores y Enfoque Financiero:**
   - La repeticion de lineas tecnicas y de hacedores de politica economica en ambas gestiones se asocia a una recurrencia en la busqueda de asistencia y negociacion de deudas con el FMI, priorizando el saneamiento monetario-cambiario de corto plazo sobre la recomposicion de los ingresos reales y productivos de la poblacion.
3. **Implicancias para el Aprendizaje Automatico:**
   - La severa distorsion nominal e institucional existente entre ambos periodos obliga a los modelos predictivos a incorporar transformaciones de escala no lineales y estratificacion robusta, dado que los cambios de comportamiento de los agentes financieros operan bajo expectativas racionales adaptadas a crisis sistematicas.